# LeetCode 3158 — Find the XOR of Numbers Which Appear Twice bằng Python

Link đề: https://leetcode.com/problems/find-the-xor-of-numbers-which-appear-twice/ · Độ khó: **Easy** · Chủ đề: Bit Manipulation, Array, Hash Table

Skeleton LeetCode:

```python
class Solution:
    def duplicateNumbersXOR(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết lại theo đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi số trong mảng xuất hiện **một lần hoặc hai lần**; ta cần **XOR của tất cả những số xuất hiện đúng hai lần**, và trả `0` nếu không có số nào xuất hiện hai lần.
2. Nắm các công cụ Python: toán tử **`^`** (XOR) và **`^=`**, tính chất **`a ^ a = 0`**, **`a ^ 0 = a`**, `set`, `dict`/`collections.Counter`, `list.count`, và mảng đếm theo giá trị.
3. Đi từ cách **so từng cặp** (`O(n²)`) → **đếm bằng `Counter`/`dict`** → **một lượt với `set` "đã thấy"** → **mảng đếm 51 ô** → **mẹo XOR hai lần** (`XOR tất cả ^ XOR các số khác nhau`).
4. Hiểu mấu chốt: vì mỗi số xuất hiện **tối đa hai lần**, "gặp lại một số đã thấy" xảy ra **đúng một lần** cho mỗi số lặp — nên chỉ cần XOR vào đáp án **ngay lúc gặp lại**.
5. Tránh các lỗi kinh điển: **XOR toàn bộ mảng** (ra XOR của các số xuất hiện **một** lần, ngược đề), khởi tạo đáp án bằng `1` thay vì `0`, dùng `+=` thay cho `^=`, và **thêm vào `set` trước khi kiểm tra**.

## 2. Đề bài & ví dụ

Cho mảng `nums`, trong đó mỗi số xuất hiện **một lần hoặc hai lần**.

Trả về **phép XOR bit** của tất cả những số xuất hiện **hai lần** trong mảng, hoặc `0` nếu không có số nào xuất hiện hai lần.

**Ví dụ 1**

- Input: `nums = [1,2,1,3]`
- Output: `1`
- Giải thích: số duy nhất xuất hiện hai lần là `1`, nên đáp án là `1`.

**Ví dụ 2**

- Input: `nums = [1,2,3]`
- Output: `0`
- Giải thích: không có số nào xuất hiện hai lần.

**Ví dụ 3**

- Input: `nums = [1,2,2,1]`
- Output: `3`
- Giải thích: `1` và `2` đều xuất hiện hai lần; `1 XOR 2 = 3`.

**Ràng buộc:** `1 <= nums.length <= 50`, `1 <= nums[i] <= 50`, mỗi số trong `nums` xuất hiện **một hoặc hai lần**.

In [1]:
from typing import List
from collections import Counter
from functools import reduce
from operator import xor
import random
import itertools

examples = [
    ([1, 2, 1, 3], 1),
    ([1, 2, 3], 0),
    ([1, 2, 2, 1], 3),
]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[1, 2, 1, 3] -> kỳ vọng 1
[1, 2, 3] -> kỳ vọng 0
[1, 2, 2, 1] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. XOR là gì? Toán tử `^`

XOR (exclusive OR, "hoặc loại trừ") làm việc **trên từng bit**: hai bit **khác nhau** cho `1`, hai bit **giống nhau** cho `0`. Trong Python, XOR là toán tử **`^`** (đừng nhầm với lũy thừa `**`). Ví dụ `1 ^ 2`: `01` XOR `10` = `11` = `3`.

In [2]:
for a, b in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    print(f"{a} ^ {b} = {a ^ b}")
print()
print(f"{1:03b} ^ {2:03b} = {1 ^ 2:03b}  ->", 1 ^ 2)
print(f"{5:03b} ^ {3:03b} = {5 ^ 3:03b}  ->", 5 ^ 3)

0 ^ 0 = 0
0 ^ 1 = 1
1 ^ 0 = 1
1 ^ 1 = 0

001 ^ 010 = 011  -> 3
101 ^ 011 = 110  -> 6


### 3.2. Ba tính chất vàng của XOR

1. **`a ^ a = 0`**: một số XOR với chính nó triệt tiêu.
2. **`a ^ 0 = a`**: `0` là **phần tử trung hòa** (XOR với `0` không đổi) — nên đáp án nên khởi tạo `ans = 0`.
3. **Giao hoán & kết hợp**: `a ^ b = b ^ a`, `(a ^ b) ^ c = a ^ (b ^ c)` — thứ tự XOR không quan trọng.

Từ đó: XOR một số **chẵn lần** thì triệt tiêu, **lẻ lần** thì còn lại đúng một bản. Ví dụ `5 ^ 5 ^ 5 = 5`.

In [3]:
a, b, c = 5, 9, 12
print(a ^ a)                       # 0
print(a ^ 0)                       # 5
print((a ^ b) == (b ^ a))          # True
print(((a ^ b) ^ c) == (a ^ (b ^ c)))   # True
print(5 ^ 5 ^ 5)                   # 5 (xuất hiện 3 lần = lẻ)
print(5 ^ 5 ^ 5 ^ 5)               # 0 (xuất hiện 4 lần = chẵn)

0
5
True
True
5
0


### 3.3. `^=` và gom nhiều số bằng `reduce`

`ans ^= x` nghĩa là `ans = ans ^ x`. Để XOR cả danh sách ta khởi tạo `ans = 0` rồi `ans ^= x` cho từng `x`. Có thể dùng `functools.reduce(operator.xor, nums, 0)`: số `0` cuối cùng là giá trị khởi đầu (vẫn đúng khi danh sách rỗng).

In [4]:
nums = [1, 2, 3]
ans = 0
for x in nums:
    ans ^= x
    print("sau khi XOR", x, "->", ans)
print(reduce(xor, nums, 0))        # 0 : 1 ^ 2 ^ 3 = 0
print(reduce(xor, [], 0))          # 0 : danh sách rỗng vẫn an toàn

sau khi XOR 1 -> 1
sau khi XOR 2 -> 3
sau khi XOR 3 -> 0
0
0


### 3.4. `set` — tập hợp các giá trị khác nhau

`set` chỉ giữ **một bản** cho mỗi giá trị và kiểm tra `x in s` rất nhanh (trung bình `O(1)`). `set(nums)` cho tập các số khác nhau. `s.add(x)` thêm phần tử.

In [5]:
s = set()
for x in [1, 2, 1, 3]:
    print(f"x={x}: đã có trong set? {x in s}")
    s.add(x)
print(s)                       # {1, 2, 3}
print(set([1, 2, 2, 1]))       # {1, 2}

x=1: đã có trong set? False
x=2: đã có trong set? False
x=1: đã có trong set? True
x=3: đã có trong set? False
{1, 2, 3}
{1, 2}


### 3.5. Đếm số lần xuất hiện: `Counter`, `dict` và `list.count`

`Counter(nums)` trả về từ điển `{giá trị: số lần xuất hiện}`. Với list ta cũng có `nums.count(x)` (quét cả mảng, `O(n)` mỗi lần gọi). Mảng đếm `cnt = [0] * 51` cũng làm được vì `nums[i] <= 50`: `cnt[x] += 1`.

In [6]:
nums = [1, 2, 1, 3]
print(Counter(nums))                       # Counter({1: 2, 2: 1, 3: 1})
print(nums.count(1), nums.count(9))        # 2 0
cnt = [0] * 6
for x in nums:
    cnt[x] += 1
print(cnt)                                 # [0, 2, 1, 1, 0, 0]

Counter({1: 2, 2: 1, 3: 1})
2 0
[0, 2, 1, 1, 0, 0]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `nums = [1, 2, 2, 1, 3]` (ví dụ tự dựng), ta đếm mỗi số xuất hiện mấy lần:

| số | số lần | xuất hiện hai lần? |
|---|---|---|
| 1 | 2 | **có** |
| 2 | 2 | **có** |
| 3 | 1 | không |

Các số "hai lần" là `1` và `2`, XOR của chúng `1 ^ 2 = 3`. Số `3` chỉ xuất hiện một lần nên **không** được tính.

Câu hỏi trực giác: *làm sao biết "số này xuất hiện hai lần" mà không đếm mọi số trước?* Khi ta duyệt từ trái sang phải, một số lặp sẽ bị **gặp lại** lần thứ hai. Vì mỗi số tối đa hai lần, lần gặp lại đó **chỉ xảy ra đúng một lần** cho mỗi số lặp.

In [7]:
nums = [1, 2, 2, 1, 3]
cnt = Counter(nums)
twice = [v for v, c in cnt.items() if c == 2]
print("các số xuất hiện hai lần:", twice)
ans = 0
for v in twice:
    ans ^= v
print("XOR =", ans)                # 1 ^ 2 = 3

các số xuất hiện hai lần: [1, 2]
XOR = 3


## 5. Cách 1 — Brute Force / baseline: so từng cặp `(i, j)`

**Ý tưởng.** Với mọi cặp `i < j`, nếu `nums[i] == nums[j]` thì số đó xuất hiện (ít nhất) hai lần. Vì mỗi số xuất hiện **tối đa hai lần**, mỗi số lặp tạo ra **đúng một** cặp bằng nhau, nên chỉ cần XOR `nums[i]` vào đáp án mỗi khi tìm thấy một cặp bằng nhau.

In [8]:
def xor_duplicates_pairs(nums: List[int]) -> int:
    ans = 0
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] == nums[j]:        # mỗi số lặp có đúng một cặp (i, j)
                ans ^= nums[i]
    return ans


for nums, expected in examples:
    print(nums, "->", xor_duplicates_pairs(nums), "(kỳ vọng", expected, ")")

[1, 2, 1, 3] -> 1 (kỳ vọng 1 )
[1, 2, 3] -> 0 (kỳ vọng 0 )
[1, 2, 2, 1] -> 3 (kỳ vọng 3 )


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`. Hai vòng lồng ⇒ **Time `O(n²)`**, **Space `O(1)`**. Với `n <= 50` chỉ khoảng `1225` cặp — rất nhanh, nhưng cách này mất tính tổng quát khi `n` lớn.

**Nút thắt:** để biết "số này đã xuất hiện trước đó chưa" ta quét lại mảng. Nếu có một "bộ nhớ" các số đã thấy thì chỉ cần tra `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Ghi nhớ các số đã thấy.** Duyệt từ trái sang phải, giữ một `set` tên `seen`:

- Nếu `x` **đã có** trong `seen` ⇒ đây là **lần xuất hiện thứ hai** của `x` ⇒ `ans ^= x`.
- Nếu chưa ⇒ thêm `x` vào `seen`.

Vì mỗi số tối đa hai lần, mỗi số lặp "bị bắt" **đúng một lần**, không bao giờ XOR hai lần (nếu cho phép ba lần thì lần thứ ba sẽ XOR thêm và làm sai — nhưng đề không cho). Cùng ý này có thể thực hiện bằng cách **đếm** rồi XOR các số có đếm bằng `2`, hoặc dùng mảng đếm.

In [9]:
nums = [1, 2, 2, 1, 3]
seen = set()
ans = 0
for x in nums:
    if x in seen:
        ans ^= x
        print(f"x={x}: đã thấy -> ans ^= {x} -> ans = {ans}")
    else:
        seen.add(x)
        print(f"x={x}: lần đầu -> thêm vào seen = {sorted(seen)}")
print("đáp án:", ans)

x=1: lần đầu -> thêm vào seen = [1]
x=2: lần đầu -> thêm vào seen = [1, 2]
x=2: đã thấy -> ans ^= 2 -> ans = 2
x=1: đã thấy -> ans ^= 1 -> ans = 3
x=3: lần đầu -> thêm vào seen = [1, 2, 3]
đáp án: 3


## 7. Các cách tiếp theo

### Cách 2 — Đếm bằng `Counter` rồi XOR các số có đếm bằng `2`

**Giải thích từng đoạn.**

1. `cnt = Counter(nums)`: số lần xuất hiện của mỗi giá trị.
2. `for v, c in cnt.items()`: duyệt từng (giá trị, số lần).
3. `if c == 2: ans ^= v`: chỉ XOR những giá trị xuất hiện hai lần.

In [10]:
def xor_duplicates_counter(nums: List[int]) -> int:
    cnt = Counter(nums)
    ans = 0
    for v, c in cnt.items():
        if c == 2:
            ans ^= v
    return ans


for nums, expected in examples:
    print(nums, "->", xor_duplicates_counter(nums), "(kỳ vọng", expected, ")")

[1, 2, 1, 3] -> 1 (kỳ vọng 1 )
[1, 2, 3] -> 0 (kỳ vọng 0 )
[1, 2, 2, 1] -> 3 (kỳ vọng 3 )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionCounter:
    def duplicateNumbersXOR(self, nums: List[int]) -> int:
        ans = 0
        for v, c in Counter(nums).items():
            if c == 2:
                ans ^= v
        return ans


print(SolutionCounter().duplicateNumbersXOR([1, 2, 2, 1]))      # 3

3


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(n)`** (từ điển đếm).

### Cách 3 — Một lượt với `set` "đã thấy"

**Giải thích.** Đây là ý tưởng ở mục 6: duyệt một lần, gặp `x` đã trong `seen` thì `ans ^= x`, ngược lại `seen.add(x)`. Chú ý thứ tự: **kiểm tra trước, thêm sau** (nếu thêm trước thì `x in seen` luôn đúng).

In [12]:
def xor_duplicates_seen(nums: List[int]) -> int:
    seen = set()
    ans = 0
    for x in nums:
        if x in seen:
            ans ^= x          # lần xuất hiện thứ hai
        else:
            seen.add(x)       # lần đầu
    return ans


for nums, expected in examples:
    print(nums, "->", xor_duplicates_seen(nums), "(kỳ vọng", expected, ")")

[1, 2, 1, 3] -> 1 (kỳ vọng 1 )
[1, 2, 3] -> 0 (kỳ vọng 0 )
[1, 2, 2, 1] -> 3 (kỳ vọng 3 )


In [13]:
# Code nộp LeetCode cho Cách 3
class SolutionSeen:
    def duplicateNumbersXOR(self, nums: List[int]) -> int:
        seen = set()
        ans = 0
        for x in nums:
            if x in seen:
                ans ^= x
            else:
                seen.add(x)
        return ans


print(SolutionSeen().duplicateNumbersXOR([1, 2, 1, 3]))         # 1

1


**Độ phức tạp Cách 3:** **Time `O(n)`** (một lượt), **Space `O(n)`** (`set`).

### Cách 4 — Mảng đếm 51 ô (vì `nums[i] <= 50`)

**Giải thích.** Giá trị chỉ từ `1` đến `50`, nên thay `Counter` bằng danh sách `cnt = [0] * 51`: `cnt[x]` là số lần xuất hiện của `x`. Sau khi đếm, duyệt `v` từ `1` đến `50`, nếu `cnt[v] == 2` thì `ans ^= v`. Không cần dict, không băm; kích thước cố định.

In [14]:
def xor_duplicates_array(nums: List[int]) -> int:
    cnt = [0] * 51                       # chỉ số 0..50 (giá trị 1..50)
    for x in nums:
        cnt[x] += 1
    ans = 0
    for v in range(1, 51):
        if cnt[v] == 2:
            ans ^= v
    return ans


for nums, expected in examples:
    print(nums, "->", xor_duplicates_array(nums), "(kỳ vọng", expected, ")")

[1, 2, 1, 3] -> 1 (kỳ vọng 1 )
[1, 2, 3] -> 0 (kỳ vọng 0 )
[1, 2, 2, 1] -> 3 (kỳ vọng 3 )


In [15]:
# Code nộp LeetCode cho Cách 4
class SolutionArray:
    def duplicateNumbersXOR(self, nums: List[int]) -> int:
        cnt = [0] * 51
        for x in nums:
            cnt[x] += 1
        ans = 0
        for v in range(1, 51):
            if cnt[v] == 2:
                ans ^= v
        return ans


print(SolutionArray().duplicateNumbersXOR([1, 2, 2, 1]))       # 3

3


**Độ phức tạp Cách 4:** **Time `O(n + 50)` = `O(n)`**, **Space `O(50) = O(1)`** (mảng cố định, không phụ thuộc `n`).

### Cách 5 — Mẹo XOR hai lần: `XOR(tất cả) ^ XOR(các số khác nhau)`

**Giải thích.** Gọi `A` là XOR **tất cả** phần tử, `B` là XOR **các số khác nhau** (`set(nums)`).

- Một số xuất hiện **một lần**: có mặt `1` lần trong `A` và `1` lần trong `B` ⇒ trong `A ^ B` xuất hiện `2` lần ⇒ **triệt tiêu**.
- Một số xuất hiện **hai lần**: có mặt `2` lần trong `A` (triệt tiêu, góp `0`) và `1` lần trong `B` ⇒ trong `A ^ B` còn lại đúng **một bản**.

Vậy `A ^ B` chính là XOR của các số xuất hiện hai lần. Hai dòng `reduce`, không cần `if` nào. (Mẹo này chỉ đúng nhờ ràng buộc "tối đa hai lần".)

In [16]:
def xor_duplicates_trick(nums: List[int]) -> int:
    all_xor = reduce(xor, nums, 0)               # A
    distinct_xor = reduce(xor, set(nums), 0)     # B
    return all_xor ^ distinct_xor


for nums, expected in examples:
    print(nums, "->", xor_duplicates_trick(nums), "(kỳ vọng", expected, ")")

[1, 2, 1, 3] -> 1 (kỳ vọng 1 )
[1, 2, 3] -> 0 (kỳ vọng 0 )
[1, 2, 2, 1] -> 3 (kỳ vọng 3 )


In [17]:
# Code nộp LeetCode cho Cách 5
class SolutionTrick:
    def duplicateNumbersXOR(self, nums: List[int]) -> int:
        ans = 0
        for x in nums:
            ans ^= x
        for x in set(nums):
            ans ^= x
        return ans


print(SolutionTrick().duplicateNumbersXOR([1, 2, 2, 1, 3]))    # 3

3


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(n)`** (`set(nums)`).

## 8. TRACE từng bước

Ta trace Cách 3 (`set` đã thấy) trên `nums = [1, 2, 1, 3]` và `nums = [1, 2, 2, 1]`, và Cách 5 (mẹo XOR hai lần) trên `nums = [1, 2, 2, 1, 3]`.

In [18]:
def trace_seen(nums: List[int]) -> int:
    print(f"nums = {nums}")
    seen = set()
    ans = 0
    for i, x in enumerate(nums):
        if x in seen:
            ans ^= x
            print(f"  i={i}: x={x} ĐÃ thấy -> ans ^= {x} -> ans = {ans} (nhị phân {ans:03b}), seen = {sorted(seen)}")
        else:
            seen.add(x)
            print(f"  i={i}: x={x} lần đầu  -> thêm vào seen = {sorted(seen)}, ans = {ans}")
    return ans


assert trace_seen([1, 2, 1, 3]) == 1
assert trace_seen([1, 2, 2, 1]) == 3
assert trace_seen([1, 2, 3]) == 0

nums = [1, 2, 1, 3]
  i=0: x=1 lần đầu  -> thêm vào seen = [1], ans = 0
  i=1: x=2 lần đầu  -> thêm vào seen = [1, 2], ans = 0
  i=2: x=1 ĐÃ thấy -> ans ^= 1 -> ans = 1 (nhị phân 001), seen = [1, 2]
  i=3: x=3 lần đầu  -> thêm vào seen = [1, 2, 3], ans = 1
nums = [1, 2, 2, 1]
  i=0: x=1 lần đầu  -> thêm vào seen = [1], ans = 0
  i=1: x=2 lần đầu  -> thêm vào seen = [1, 2], ans = 0
  i=2: x=2 ĐÃ thấy -> ans ^= 2 -> ans = 2 (nhị phân 010), seen = [1, 2]
  i=3: x=1 ĐÃ thấy -> ans ^= 1 -> ans = 3 (nhị phân 011), seen = [1, 2]
nums = [1, 2, 3]
  i=0: x=1 lần đầu  -> thêm vào seen = [1], ans = 0
  i=1: x=2 lần đầu  -> thêm vào seen = [1, 2], ans = 0
  i=2: x=3 lần đầu  -> thêm vào seen = [1, 2, 3], ans = 0


In [19]:
def trace_trick(nums: List[int]) -> int:
    print(f"nums = {nums}")
    all_xor = 0
    for x in nums:
        all_xor ^= x
        print(f"  XOR tất cả: sau {x} -> {all_xor}")
    distinct_xor = 0
    for x in sorted(set(nums)):
        distinct_xor ^= x
        print(f"  XOR các số khác nhau: sau {x} -> {distinct_xor}")
    ans = all_xor ^ distinct_xor
    print(f"  A = {all_xor}, B = {distinct_xor}, A ^ B = {ans}")
    return ans


assert trace_trick([1, 2, 2, 1, 3]) == 3

nums = [1, 2, 2, 1, 3]
  XOR tất cả: sau 1 -> 1
  XOR tất cả: sau 2 -> 3
  XOR tất cả: sau 2 -> 1
  XOR tất cả: sau 1 -> 0
  XOR tất cả: sau 3 -> 3
  XOR các số khác nhau: sau 1 -> 1
  XOR các số khác nhau: sau 2 -> 3
  XOR các số khác nhau: sau 3 -> 0
  A = 3, B = 0, A ^ B = 3


Bảng trace Cách 3 với `nums = [1, 2, 2, 1]`:

| i | `x` | `x` đã trong `seen`? | hành động | `seen` sau | `ans` sau |
|---|---|---|---|---|---|
| 0 | 1 | không | thêm 1 | {1} | 0 |
| 1 | 2 | không | thêm 2 | {1, 2} | 0 |
| 2 | 2 | **có** | `ans ^= 2` | {1, 2} | 2 |
| 3 | 1 | **có** | `ans ^= 1` | {1, 2} | 3 |

Đáp án `3` (`2 ^ 1 = 3`).

Bảng trace Cách 5 với `nums = [1, 2, 2, 1, 3]`: `A = 1 ^ 2 ^ 2 ^ 1 ^ 3 = 3`; `B = 1 ^ 2 ^ 3 = 0`; `A ^ B = 3`. Số `3` (xuất hiện một lần) có mặt trong cả `A` và `B` nên triệt tiêu; số `1`, `2` (hai lần) chỉ còn lại từ `B`: `1 ^ 2 = 3`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (`set` đã thấy, một lượt): dễ hiểu, `O(n)`.

In [20]:
class Solution:
    def duplicateNumbersXOR(self, nums: List[int]) -> int:
        seen = set()                       # các số đã gặp
        ans = 0                            # 0 là phần tử trung hòa của XOR
        for x in nums:
            if x in seen:                  # lần xuất hiện thứ hai
                ans ^= x
            else:
                seen.add(x)                # lần đầu: ghi nhớ
        return ans


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.duplicateNumbersXOR(nums), "(kỳ vọng", expected, ")")

[1, 2, 1, 3] -> 1 (kỳ vọng 1 )
[1, 2, 3] -> 0 (kỳ vọng 0 )
[1, 2, 2, 1] -> 3 (kỳ vọng 3 )


## 10. Vì sao thuật toán đúng

- **Mỗi số lặp được XOR đúng một lần:** đề đảm bảo mỗi số xuất hiện **một hoặc hai lần**. Một số chỉ xuất hiện một lần không bao giờ thỏa `x in seen` ⇒ không bị XOR. Một số xuất hiện hai lần thỏa `x in seen` đúng ở lần thứ hai ⇒ bị XOR đúng một lần.
- **Khởi tạo đúng:** `ans = 0` là phần tử trung hòa: nếu không có số lặp thì `ans` giữ nguyên `0` — khớp yêu cầu "trả `0` nếu không có số nào xuất hiện hai lần".
- **Thứ tự không quan trọng:** XOR giao hoán và kết hợp, nên XOR các số lặp theo thứ tự lần gặp lại cho cùng kết quả với XOR theo thứ tự bất kỳ.
- **Mẹo hai lần (Cách 5):** đếm số lần mỗi giá trị `v` góp vào `A ^ B`: `cnt(v) + 1` lần. `cnt(v) = 1` ⇒ `2` lần ⇒ chẵn ⇒ triệt tiêu; `cnt(v) = 2` ⇒ `3` lần ⇒ lẻ ⇒ còn một bản. Đúng đề.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (`n <= 50`), `V = 50` là giá trị lớn nhất.

| Cách | Time | Space |
|---|---|---|
| 1. So từng cặp | `O(n²)` | `O(1)` |
| 2. `Counter` | `O(n)` | `O(n)` |
| 3. `set` đã thấy (một lượt) | `O(n)` | `O(n)` |
| 4. Mảng đếm 51 ô | `O(n + V)` | `O(V) = O(1)` |
| 5. XOR hai lần | `O(n)` | `O(n)` |

Mọi cách đều tức thì với `n <= 50`; điểm khác biệt nằm ở tính tổng quát khi `n` lớn.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [21]:
sol = Solution()
impls = {
    "pairs": xor_duplicates_pairs,
    "counter": xor_duplicates_counter,
    "seen": xor_duplicates_seen,
    "array": xor_duplicates_array,
    "trick": xor_duplicates_trick,
    "leetcode": sol.duplicateNumbersXOR,
}


def check_all(nums, expected, label):
    for key, fn in impls.items():
        actual = fn(list(nums))
        assert actual == expected, (label, key, nums, actual, expected)
    print("PASS", label)


for idx, (nums, expected) in enumerate(examples, 1):
    check_all(nums, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [22]:
check_all([7], 0, "một phần tử")
check_all([5, 5], 5, "hai phần tử bằng nhau")
check_all([1, 2], 0, "hai phần tử khác nhau")
check_all([50, 50], 50, "giá trị lớn nhất, lặp")
check_all([1, 1], 1, "giá trị nhỏ nhất, lặp")
check_all([3, 5, 3, 5], 3 ^ 5, "hai số lặp: 3 ^ 5 = 6")
check_all([1, 2, 3, 1, 2, 3], 1 ^ 2 ^ 3, "ba số lặp: kết quả 0")
check_all([1, 2, 3, 4, 1, 2, 3, 4], 1 ^ 2 ^ 3 ^ 4, "bốn số lặp: kết quả 4")
check_all([8, 1, 8, 2, 3], 8, "số lặp cách xa nhau")
check_all(list(range(1, 51)), 0, "50 số khác nhau")
check_all(list(range(1, 26)) * 2, reduce(xor, range(1, 26), 0), "25 số, mỗi số hai lần")

PASS một phần tử
PASS hai phần tử bằng nhau
PASS hai phần tử khác nhau
PASS giá trị lớn nhất, lặp
PASS giá trị nhỏ nhất, lặp
PASS hai số lặp: 3 ^ 5 = 6
PASS ba số lặp: kết quả 0
PASS bốn số lặp: kết quả 4
PASS số lặp cách xa nhau
PASS 50 số khác nhau
PASS 25 số, mỗi số hai lần


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **XOR toàn bộ mảng** (ra XOR của các số xuất hiện **một** lần); (b) **thêm vào `set` trước khi kiểm tra** (số nào cũng bị coi là "đã thấy").

In [23]:
def wrong_xor_all(nums):
    return reduce(xor, nums, 0)                      # SAI: ra XOR của số xuất hiện một lần


def wrong_add_first(nums):
    seen = set()
    ans = 0
    for x in nums:
        seen.add(x)                                  # SAI: thêm trước
        if x in seen:
            ans ^= x
    return ans


print(wrong_xor_all([3, 5, 3, 7]), "!=", xor_duplicates_seen([3, 5, 3, 7]))        # 5 ^ 7 = 2 so với 3
print(wrong_add_first([2, 5, 6]), "!=", xor_duplicates_seen([2, 5, 6]))
assert wrong_xor_all([3, 5, 3, 7]) != xor_duplicates_seen([3, 5, 3, 7])
assert wrong_add_first([2, 5, 6]) != xor_duplicates_seen([2, 5, 6])

2 != 3
1 != 0


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn

Đầu vào hợp lệ: độ dài `1..50`, giá trị `1..50`, mỗi giá trị xuất hiện một hoặc hai lần. Ta sinh: chọn ngẫu nhiên tập giá trị, với mỗi giá trị chọn một hay hai lần, rồi xáo trộn. Đối chiếu với một "oracle" viết bằng `list.count`. Ngoài ra **vét cạn** mọi cách chọn "một lần / hai lần" cho các giá trị `1..6` (mỗi giá trị có 3 trạng thái: vắng, một lần, hai lần).

In [24]:
def oracle(nums):
    ans = 0
    for v in set(nums):
        if nums.count(v) == 2:
            ans ^= v
    return ans


def valid(nums):
    return 1 <= len(nums) <= 50 and all(1 <= x <= 50 for x in nums) and all(nums.count(x) in (1, 2) for x in set(nums))


rng = random.Random(3158)
for trial in range(4000):
    values = rng.sample(range(1, 51), rng.randint(1, 25))
    nums = []
    for v in values:
        nums += [v] * rng.choice([1, 2])
    rng.shuffle(nums)
    assert valid(nums), nums
    ref = oracle(nums)
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums)
print("PASS 4000 mảng ngẫu nhiên đúng ràng buộc")

count = 0
for states in itertools.product([0, 1, 2], repeat=6):            # trạng thái mỗi giá trị 1..6
    base = []
    for v, st in zip(range(1, 7), states):
        base += [v] * st
    if not base:
        continue
    for perm_seed in range(3):
        nums = list(base)
        random.Random(perm_seed).shuffle(nums)
        ref = oracle(nums)
        for key, fn in impls.items():
            assert fn(list(nums)) == ref, (key, nums)
        count += 1
print("PASS vét cạn", count, "mảng (mọi trạng thái của giá trị 1..6)")

PASS 4000 mảng ngẫu nhiên đúng ràng buộc


PASS vét cạn 2184 mảng (mọi trạng thái của giá trị 1..6)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: XOR toàn bộ mảng

In [25]:
def sai_xor_het(nums):
    ans = 0
    for x in nums:
        ans ^= x
    return ans


print("sai:", sai_xor_het([3, 5, 3, 7]), "| đúng:", Solution().duplicateNumbersXOR([3, 5, 3, 7]))
print("sai:", sai_xor_het([4, 9, 4]), "| đúng:", Solution().duplicateNumbersXOR([4, 9, 4]))
print("sai:", sai_xor_het([1, 2, 3]), "| đúng:", Solution().duplicateNumbersXOR([1, 2, 3]), "(trùng ngẫu nhiên vì 1 ^ 2 ^ 3 = 0)")

sai: 2 | đúng: 3
sai: 9 | đúng: 4
sai: 0 | đúng: 0 (trùng ngẫu nhiên vì 1 ^ 2 ^ 3 = 0)


**Vì sao sai:** XOR toàn bộ làm các số xuất hiện **hai lần triệt tiêu**, thứ còn lại là XOR của các số xuất hiện **một lần** — đúng ngược đề. **Cách sửa:** chỉ XOR các số được xác định là xuất hiện hai lần (khi gặp lại hoặc có đếm bằng `2`).

### Lỗi 2: Khởi tạo `ans = 1`

In [26]:
def sai_khoi_tao_1(nums):
    seen = set()
    ans = 1                              # SAI: 1 không phải phần tử trung hòa của XOR
    for x in nums:
        if x in seen:
            ans ^= x
        else:
            seen.add(x)
    return ans


print("sai:", sai_khoi_tao_1([1, 2, 3]), "| đúng:", Solution().duplicateNumbersXOR([1, 2, 3]))
print("sai:", sai_khoi_tao_1([1, 2, 1, 3]), "| đúng:", Solution().duplicateNumbersXOR([1, 2, 1, 3]))

sai: 1 | đúng: 0
sai: 0 | đúng: 1


**Vì sao sai:** `a ^ 0 = a` nhưng `a ^ 1 ≠ a`. Khởi tạo `1` làm lệch mọi kết quả (và ca không có số lặp trả `1` thay vì `0`). **Cách sửa:** khởi tạo `ans = 0` (phần tử trung hòa của XOR; tương tự `0` cho phép cộng, `1` cho phép nhân).

### Lỗi 3: Dùng `+=` hoặc `|=` thay vì `^=`

In [27]:
def sai_dung_cong(nums):
    seen = set()
    ans = 0
    for x in nums:
        if x in seen:
            ans += x                     # SAI: cộng chứ không phải XOR
        else:
            seen.add(x)
    return ans


print("sai:", sai_dung_cong([1, 2, 2, 1]), "| đúng:", Solution().duplicateNumbersXOR([1, 2, 2, 1]))       # 3 và 3 (trùng ngẫu nhiên)
print("sai:", sai_dung_cong([3, 5, 3, 5]), "| đúng:", Solution().duplicateNumbersXOR([3, 5, 3, 5]))       # 8 và 6

sai: 3 | đúng: 3
sai: 8 | đúng: 6


**Vì sao sai:** cộng và XOR chỉ trùng nhau khi hai số **không có bit chung** (`1 + 2 = 1 ^ 2 = 3`), còn `3` và `5` chung bit thấp nhất nên `3 + 5 = 8` mà `3 ^ 5 = 6`. Một ví dụ đề trùng ngẫu nhiên không chứng minh code đúng — hãy thử thêm ví dụ khác. **Cách sửa:** dùng `^=`.

### Lỗi 4: Thêm vào `set` trước khi kiểm tra

In [28]:
def sai_them_truoc(nums):
    seen = set()
    ans = 0
    for x in nums:
        seen.add(x)
        if x in seen:                    # SAI: luôn đúng vì vừa thêm xong
            ans ^= x
    return ans


print("sai:", sai_them_truoc([2, 5, 6]), "| đúng:", Solution().duplicateNumbersXOR([2, 5, 6]))
print("sai:", sai_them_truoc([3, 5, 3, 7]), "| đúng:", Solution().duplicateNumbersXOR([3, 5, 3, 7]))

sai: 1 | đúng: 0
sai: 2 | đúng: 3


**Vì sao sai:** sau `seen.add(x)`, điều kiện `x in seen` luôn đúng, nên mọi phần tử đều bị XOR (thực chất là XOR toàn mảng, quay về Lỗi 1). **Cách sửa:** **kiểm tra trước, thêm sau** (`if x in seen: ... else: seen.add(x)`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Xuất hiện hai lần / một lần**" + "XOR" ⇒ nghĩ ngay tới `a ^ a = 0`. Nhưng đọc kỹ đề hỏi **số lặp** hay **số đơn**: bài *Single Number* thì XOR cả mảng; bài này ngược lại.
- "**Đã gặp chưa?**" ⇒ `set` (kiểm tra `in` `O(1)`) hoặc `Counter`. Khi giá trị nhỏ và biết cận (≤ 50) ⇒ mảng đếm.
- Phần tử trung hòa: XOR là `0`, cộng là `0`, nhân là `1`. Khởi tạo sai là lỗi kinh điển.
- Ràng buộc "tối đa hai lần" là chìa khóa: nó đảm bảo "gặp lại" chỉ xảy ra một lần. Nếu đề cho phép nhiều lần hơn, phải đổi sang đếm `== 2` hoặc `>= 2` tùy yêu cầu.
- Dùng `bin(x)` / `f"{x:08b}"` để nhìn bit khi nghi ngờ.

## 15. Biến thể của bài

1. **Số xuất hiện hai lần bất kỳ lần** (`>= 2`): XOR các giá trị có đếm `>= 2` (mỗi giá trị chỉ một lần).
2. **XOR các số xuất hiện đúng một lần** (ngược lại bài này): chính là `XOR toàn bộ` khi mọi số tối đa hai lần.
3. **Cộng/nhân thay cho XOR:** tổng/tích các số xuất hiện hai lần.
4. **Bitmask:** dùng một số nguyên làm "set": bit thứ `x` bằng `1` nếu đã thấy `x` (hợp với giá trị `<= 50` vì Python có số nguyên lớn tùy ý).
5. Bài gần gũi: **Single Number (136)**, **Find the Duplicate Number (287)**, **Contains Duplicate (217)**, **Single Number III (260)**.

In [29]:
def xor_appearing_at_least_twice(nums):
    return reduce(xor, [v for v, c in Counter(nums).items() if c >= 2], 0)


def xor_appearing_once(nums):
    return reduce(xor, [v for v, c in Counter(nums).items() if c == 1], 0)


def sum_appearing_twice(nums):
    return sum(v for v, c in Counter(nums).items() if c == 2)


def xor_duplicates_bitmask(nums):
    mask = 0                                   # bit x = 1 nghĩa là đã thấy x
    ans = 0
    for x in nums:
        if (mask >> x) & 1:
            ans ^= x
        else:
            mask |= 1 << x
    return ans


print(xor_appearing_at_least_twice([1, 1, 1, 2, 2, 3]))      # 1 ^ 2 = 3
print(xor_appearing_once([1, 2, 1, 3]))                      # 2 ^ 3 = 1
print(sum_appearing_twice([3, 5, 3, 5, 7]))                  # 8
print(xor_duplicates_bitmask([1, 2, 2, 1, 3]))               # 3
assert xor_appearing_at_least_twice([1, 1, 1, 2, 2, 3]) == 3
assert xor_appearing_once([1, 2, 1, 3]) == 2 ^ 3
assert sum_appearing_twice([3, 5, 3, 5, 7]) == 8
for nums, expected in examples:
    assert xor_duplicates_bitmask(nums) == expected
assert xor_appearing_once([1, 2, 2, 1, 3]) == reduce(xor, [1, 2, 2, 1, 3], 0) == 3

3
1
8
3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. So từng cặp | mỗi cặp bằng nhau `(i, j)` XOR vào đáp án | `O(n²)` | `O(1)` | `n` rất nhỏ, không cần cấu trúc phụ |
| 2. `Counter` | đếm rồi XOR giá trị có đếm bằng 2 | `O(n)` | `O(n)` | đề đổi điều kiện đếm (`>= 2`, `== 3`, ...) |
| 3. `set` đã thấy | gặp lại thì XOR | `O(n)` | `O(n)` | **nên nộp**: một lượt, dễ hiểu |
| 4. Mảng đếm 51 ô | `cnt[x] += 1`, rồi XOR `cnt[v] == 2` | `O(n + V)` | `O(1)` | giá trị nhỏ, biết cận |
| 5. XOR hai lần | `XOR(tất cả) ^ XOR(set)` | `O(n)` | `O(n)` | thích mẹo bit, code ngắn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `6 ^ 6`, `9 ^ 0`, `5 ^ 3` , `set([4, 4, 7, 4])`, `Solution().duplicateNumbersXOR([2, 2])`.

**Bài 2 (tính tay).** Với `nums = [4, 6, 4, 9, 6, 1]`: lập bảng đếm từng số, liệt kê các số xuất hiện hai lần và tính XOR của chúng (viết nhị phân).

**Bài 3 (điền TODO).** Hoàn thành `duplicate_xor_todo(nums)` theo Cách 3 (`set` đã thấy).

In [30]:
def duplicate_xor_todo(nums):
    # TODO 1: tạo set rỗng seen và ans = 0
    # TODO 2: duyệt từng x: nếu x đã có trong seen thì ans ^= x, ngược lại thêm x vào seen
    # TODO 3: trả về ans
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `duplicate_xor_sorted(nums)`: **sắp xếp** `nums` trước, rồi so sánh phần tử với phần tử đứng ngay trước nó (`nums[i] == nums[i - 1]`) để XOR. Không dùng `set`/`Counter`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `duplicates_list(nums)` trả về **danh sách tăng dần** các số xuất hiện hai lần. (b) Viết `duplicate_xor_any_count(nums)` cho mảng mà mỗi số có thể xuất hiện **nhiều lần** (không giới hạn): trả XOR của các giá trị xuất hiện **ít nhất hai lần** (mỗi giá trị một lần).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `6 ^ 6 = 0`; `9 ^ 0 = 9`; `5 ^ 3 = 6` (`101 ^ 011 = 110`); `set([4, 4, 7, 4]) = {4, 7}`; `[2, 2]` có số `2` lặp nên đáp án `2`. Bài 3: `if x in seen: ans ^= x else: seen.add(x)`.
- **Mức 2 (vừa):** Bài 2: `4` xuất hiện 2 lần, `6` xuất hiện 2 lần, `9` và `1` mỗi số một lần; XOR = `4 ^ 6 = 2` (`100 ^ 110 = 010`). Bài 4: sau `nums.sort()`, các số bằng nhau đứng cạnh nhau; nếu `nums[i] == nums[i - 1]` thì `ans ^= nums[i]` (vì tối đa hai lần nên không bị XOR hai lần).
- **Mức 3 (sâu):** Bài 5(a): `sorted(v for v, c in Counter(nums).items() if c == 2)`. 5(b): với số lần tùy ý không thể dùng "gặp lại" (sẽ XOR nhiều lần); dùng `Counter` và điều kiện `c >= 2`, hoặc một `set` thứ hai `dup` để mỗi giá trị chỉ XOR một lần (`dup` là các giá trị đã XOR).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert 6 ^ 6 == 0
assert 9 ^ 0 == 9
assert 5 ^ 3 == 6
assert set([4, 4, 7, 4]) == {4, 7}
assert Solution().duplicateNumbersXOR([2, 2]) == 2

# Bài 2
cnt2 = Counter([4, 6, 4, 9, 6, 1])
assert cnt2 == Counter({4: 2, 6: 2, 9: 1, 1: 1})
assert Solution().duplicateNumbersXOR([4, 6, 4, 9, 6, 1]) == 4 ^ 6 == 2


# Bài 3
def duplicate_xor_todo_ans(nums):
    seen = set()
    ans = 0
    for x in nums:
        if x in seen:
            ans ^= x
        else:
            seen.add(x)
    return ans


for nums, expected in examples:
    assert duplicate_xor_todo_ans(nums) == expected


# Bài 4
def duplicate_xor_sorted(nums):
    nums = sorted(nums)
    ans = 0
    for i in range(1, len(nums)):
        if nums[i] == nums[i - 1]:
            ans ^= nums[i]
    return ans


for nums, expected in examples:
    assert duplicate_xor_sorted(nums) == expected


# Bài 5(a)
def duplicates_list(nums):
    return sorted(v for v, c in Counter(nums).items() if c == 2)


assert duplicates_list([1, 2, 2, 1, 3]) == [1, 2]
assert duplicates_list([1, 2, 3]) == []


# Bài 5(b)
def duplicate_xor_any_count(nums):
    seen, dup = set(), set()
    ans = 0
    for x in nums:
        if x in seen and x not in dup:
            dup.add(x)
            ans ^= x
        seen.add(x)
    return ans


assert duplicate_xor_any_count([1, 1, 1, 2, 2, 3]) == 3
assert duplicate_xor_any_count([5, 5, 5, 5]) == 5
assert duplicate_xor_any_count([1, 2, 3]) == 0
rng2 = random.Random(2)
for _ in range(500):
    arr = [rng2.randint(1, 10) for _ in range(rng2.randint(1, 30))]
    assert duplicate_xor_any_count(arr) == xor_appearing_at_least_twice(arr)
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (gặp lại một số đã thấy thì XOR vào đáp án)

```python
class Solution:
    def duplicateNumbersXOR(self, nums: List[int]) -> int:
        seen = set()
        ans = 0
        for x in nums:
            if x in seen:
                ans ^= x
            else:
                seen.add(x)
        return ans
```

**Checklist**

- [ ] `ans` khởi tạo bằng `0` (phần tử trung hòa của XOR)?
- [ ] Dùng `^=` (XOR), không phải `+=` hay `|=`?
- [ ] **Kiểm tra** `x in seen` **trước**, `seen.add(x)` **sau**?
- [ ] Không XOR toàn bộ mảng (đó là XOR các số xuất hiện **một** lần)?
- [ ] Đã thử `[1,2,1,3]` (`1`), `[1,2,3]` (`0`), `[1,2,2,1]` (`3`), và `[3,5,3,5]` (`6`, để phân biệt XOR với cộng)?
- [ ] Skeleton đúng `class Solution` / `def duplicateNumbersXOR(self, nums: List[int]) -> int`?